In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import StratifiedKFold

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(exist_ok=True)

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

y = train[TARGET_COLUMN].copy()

X_base = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_v6 = add_title_hierarchy_features(X_base)

print("train:", train.shape)
print("X_v6:", X_v6.shape)

train: (8340, 23)
X_v6: (8340, 66)


2. Списки признаков v6

In [ ]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

EXCLUDED_COLUMNS_V6 = EXCLUDED_COLUMNS + [
    "Полное название",
]

base_numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

title_numeric_columns_v4 = [
    "Название_число_слов",
    "Название_есть_4X4",
    "Название_есть_AWD",
    "Название_есть_TURBO",
    "Название_есть_SPORT",
    "Название_есть_HYBRID",
    "Название_есть_GT",
    "Название_есть_LUXURY",
    "Название_есть_DIESEL",
]

title_hierarchy_numeric_columns = [
    "Название_мощность_kw",
    "Название_есть_мощность_kw",
    "Название_есть_AMG",
    "Название_есть_M_SPORT",
    "Название_есть_RS",
    "Название_есть_S_LINE",
    "Название_есть_GTI",
    "Название_есть_HSE",
    "Название_есть_SR5",
    "Название_есть_GXL",
    "Название_есть_LIMITED",
    "Название_есть_PREMIUM",
    "Название_есть_COMFORTLINE",
    "Название_есть_ASCENT",
    "Название_есть_ACTIVE",
    "Название_есть_ELITE",
    "Название_есть_TDI",
    "Название_есть_TSI",
    "Название_есть_TFSI",
    "Название_есть_CDI",
    "Название_есть_V6",
    "Название_есть_V8",
]

numeric_columns_v6 = (
    base_numeric_columns
    + title_numeric_columns_v4
    + title_hierarchy_numeric_columns
)

feature_columns_v5 = [
    column
    for column in X_v6.columns
    if column not in EXCLUDED_COLUMNS
]

categorical_columns_v6 = [
    column
    for column in feature_columns_v6
    if column not in numeric_columns_v6
]

assert "Полное название" not in feature_columns_v6
assert "Название_префикс_3" in feature_columns_v6
assert "Название_префикс_4" in feature_columns_v6

print("Всего признаков:", len(feature_columns_v6))
print("Числовых:", len(numeric_columns_v6))
print("Категориальных:", len(categorical_columns_v6))

Всего признаков: 58
Числовых: 40
Категориальных: 18


Матрица CatBoost и CV-фолды

In [ ]:
X_model_v5 = X_v6[
    feature_columns_v5
].copy()

categorical_columns_v5 = [
    column
    for column in feature_columns_v5
    if column not in numeric_columns_v6
]

for column in categorical_columns_v5:
    X_model_v5[column] = (
        X_model_v5[column]
        .fillna("__MISSING__")
        .astype(str)
    )

cv_target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
).to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print(X_model_v6.shape)

(8340, 58)


OOF CatBoost v6

In [ ]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


oof_v6_predictions = np.zeros(len(y))
v6_fold_results = []

for fold, (train_fold_idx, valid_fold_idx) in enumerate(
    cv.split(X_model_v5, cv_target_bins),
    start=1,
):
    print(f"\n{'=' * 60}")
    print(f"CatBoost v6 fold {fold}/5")
    print(f"{'=' * 60}")

    X_train_fold = X_model_v5.iloc[train_fold_idx].copy()
    X_valid_fold = X_model_v5.iloc[valid_fold_idx].copy()

    y_train_fold = y.iloc[train_fold_idx].copy()
    y_valid_fold = y.iloc[valid_fold_idx].copy()

    model_fold = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    model_fold.fit(
        X_train_fold,
        np.log1p(y_train_fold),
        cat_features=categorical_columns_v5,
        eval_set=(
            X_valid_fold,
            np.log1p(y_valid_fold),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    fold_predictions = np.maximum(
        np.expm1(
            model_fold.predict(X_valid_fold)
        ),
        1,
    )

    oof_v6_predictions[valid_fold_idx] = fold_predictions

    fold_mape = mape_percent(
        y_valid_fold,
        fold_predictions,
    )

    v6_fold_results.append(
        {
            "fold": fold,
            "best_iteration": model_fold.get_best_iteration(),
            "validation_mape_pct": fold_mape,
        }
    )

    print(
        f"Fold {fold} MAPE: {fold_mape:.3f}% | "
        f"best iteration: {model_fold.get_best_iteration()}"
    )


CatBoost v6 fold 1/5
0:	learn: 0.6535614	test: 0.6455360	best: 0.6455360 (0)	total: 223ms	remaining: 11m 8s
500:	learn: 0.1509100	test: 0.2110765	best: 0.2110765 (500)	total: 1m 11s	remaining: 5m 55s
1000:	learn: 0.1129219	test: 0.2023204	best: 0.2022962 (996)	total: 1m 59s	remaining: 3m 58s
1500:	learn: 0.0909581	test: 0.2000329	best: 0.2000020 (1498)	total: 2m 44s	remaining: 2m 43s
2000:	learn: 0.0746943	test: 0.1988842	best: 0.1988631 (1989)	total: 3m 29s	remaining: 1m 44s
2500:	learn: 0.0626117	test: 0.1982288	best: 0.1982203 (2480)	total: 4m 14s	remaining: 50.8s
2999:	learn: 0.0532026	test: 0.1979599	best: 0.1979483 (2972)	total: 5m 50s	remaining: 0us

bestTest = 0.1979483271
bestIteration = 2972

Shrink model to first 2973 iterations.
Fold 1 MAPE: 13.413% | best iteration: 2972

CatBoost v6 fold 2/5
0:	learn: 0.6506029	test: 0.6565337	best: 0.6565337 (0)	total: 184ms	remaining: 9m 11s
500:	learn: 0.1601411	test: 0.1933784	best: 0.1933657 (499)	total: 1m 34s	remaining: 7m 52s
100

5. Итоги и сохранение

In [5]:
v6_fold_results_df = pd.DataFrame(v6_fold_results)

oof_v6_mape = mape_percent(
    y,
    oof_v6_predictions,
)

display(v6_fold_results_df)

print(f"OOF MAPE v6: {oof_v6_mape:.3f}%")
print(
    "Mean fold MAPE:",
    f"{v6_fold_results_df['validation_mape_pct'].mean():.3f}%"
)
print(
    "Std fold MAPE:",
    f"{v6_fold_results_df['validation_mape_pct'].std():.3f}"
)

,fold,best_iteration,validation_mape_pct
0,1,2972,13.412829
1,2,2844,12.291362
2,3,2441,12.806219
3,4,2979,12.834264
4,5,2997,12.390038


OOF MAPE v6: 12.747%
Mean fold MAPE: 12.747%
Std fold MAPE: 0.444


In [ ]:
v6_oof_report = pd.DataFrame(
    {
        "car_id": train["car_id"].to_numpy(),
        "y_true": y.to_numpy(),
        "title_v6_pred": oof_v6_predictions,
        "ape_pct": (
            np.abs(y.to_numpy() - oof_v6_predictions)
            / y.to_numpy()
            * 100
        ),
    }
)

v6_oof_path = (
    REPORTS_DIR
    / "catboost_title_hierarchy_v6_oof_predictions.parquet"
)

v6_oof_report.to_parquet(
    v6_oof_path,
    index=False,
)

print("Saved:", v6_oof_path)
display(v6_oof_report.head())

Saved: C:\temp\shift_ml\reports\catboost_title_hierarchy_v6_oof_predictions.parquet


,car_id,y_true,title_v6_pred,ape_pct
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,46820.046301,20.632913
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,13640.264268,14.481102
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40781.144022,2.878914
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,74796.188103,7.004561
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,30123.132359,7.775071


Это сильное подтверждение v6.

CatBoost v4 OOF: 13.050%
CatBoost v6 OOF: 12.747%
Улучшение:        −0.303 п.п.

Std по фолдам:
v4: 0.591
v6: 0.444

Теперь собираем честный OOF-ансамбль из четырёх моделей:

Ridge
+ CatBoost v4
+ CatBoost v6
+ TF-IDF Text Ridge

Важно: даже если v4 слабее v6 отдельно, она может остаться полезной в ансамбле за счёт других ошибок.

Загрузить и объединить все OOF-прогнозы

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_percentage_error

REPORTS_DIR = PROJECT_ROOT / "reports"


def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


ridge_oof = pd.read_parquet(
    REPORTS_DIR / "ridge_oof_predictions_alpha_0_1.parquet"
)

v4_oof = pd.read_parquet(
    REPORTS_DIR / "catboost_title_v4_oof_predictions.parquet"
)

v6_oof = pd.read_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v6_oof_predictions.parquet"
)

text_oof = pd.read_parquet(
    REPORTS_DIR / "text_ridge_tfidf_oof_predictions.parquet"
)

print("Ridge:", ridge_oof.columns.tolist())
print("v4:", v4_oof.columns.tolist())
print("v6:", v6_oof.columns.tolist())
print("Text:", text_oof.columns.tolist())

Ridge: ['car_id', 'y_true', 'ridge_pred']
v4: ['car_id', 'y_true', 'title_v4_pred', 'ape_pct']
v6: ['car_id', 'y_true', 'title_v6_pred', 'ape_pct']
Text: ['car_id', 'y_true', 'text_ridge_pred', 'ape_pct']


In [8]:
oof_ensemble = (
    ridge_oof[
        ["car_id", "y_true", "ridge_pred"]
    ]
    .rename(columns={"y_true": "y_true_ridge"})
    .merge(
        v4_oof[
            ["car_id", "y_true", "title_v4_pred"]
        ].rename(columns={"y_true": "y_true_v4"}),
        on="car_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        v6_oof[
            ["car_id", "y_true", "title_v6_pred"]
        ].rename(columns={"y_true": "y_true_v6"}),
        on="car_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        text_oof[
            ["car_id", "y_true", "text_ridge_pred"]
        ].rename(columns={"y_true": "y_true_text"}),
        on="car_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(oof_ensemble) == len(train)

assert np.allclose(
    oof_ensemble["y_true_ridge"],
    oof_ensemble["y_true_v4"],
)

assert np.allclose(
    oof_ensemble["y_true_ridge"],
    oof_ensemble["y_true_v6"],
)

assert np.allclose(
    oof_ensemble["y_true_ridge"],
    oof_ensemble["y_true_text"],
)

oof_ensemble["y_true"] = oof_ensemble["y_true_ridge"]

oof_ensemble = oof_ensemble[
    [
        "car_id",
        "y_true",
        "ridge_pred",
        "title_v4_pred",
        "title_v6_pred",
        "text_ridge_pred",
    ]
].copy()

display(oof_ensemble.head())

,car_id,y_true,ridge_pred,title_v4_pred,title_v6_pred,text_ridge_pred
0,65e4207d-80c1-47a9-9ce8-51a5e5cfca5c,38812,40566.385555,47964.148615,46820.046301,41370.727136
1,331006ba-098d-4b5b-9f05-3cf9544c30ec,15950,16612.439448,13448.527652,13640.264268,17044.414914
2,43a44ee6-5293-4a42-a04f-faf3bf0967b4,41990,40958.198793,40504.712805,40781.144022,41271.250267
3,383affa3-d8da-4ad0-9bd7-91f48c1c00ac,69900,59670.249223,78713.988302,74796.188103,67829.699195
4,e100c33a-b78c-479d-8edf-5734b2bde7fd,27950,27712.258776,30937.398285,30123.132359,28567.945931


Сравнить модели по отдельности

In [9]:
prediction_columns = [
    "ridge_pred",
    "title_v4_pred",
    "title_v6_pred",
    "text_ridge_pred",
]

individual_oof_results = pd.DataFrame(
    {
        "model": [
            "Ridge alpha=0.1",
            "CatBoost title v4",
            "CatBoost hierarchy v6",
            "TF-IDF Text Ridge",
        ],
        "oof_mape_pct": [
            mape_percent(
                oof_ensemble["y_true"],
                oof_ensemble[column],
            )
            for column in prediction_columns
        ],
    }
).sort_values(
    "oof_mape_pct"
).reset_index(drop=True)

display(individual_oof_results)

,model,oof_mape_pct
0,CatBoost hierarchy v6,12.746942
1,CatBoost title v4,13.050142
2,Ridge alpha=0.1,14.468951
3,TF-IDF Text Ridge,14.903406


Подобрать OOF-веса через оптимизацию

Это безопаснее, чем перебирать веса на одном holdout: каждый OOF-прогноз получен моделью, не видевшей соответствующий объект при обучении.

In [10]:
P = oof_ensemble[
    prediction_columns
].to_numpy(dtype=float)

y_true_oof = oof_ensemble[
    "y_true"
].to_numpy(dtype=float)


def blend_oof_mape(weights: np.ndarray) -> float:
    prediction = P @ weights

    return mape_percent(
        y_true_oof,
        prediction,
    )


constraint = {
    "type": "eq",
    "fun": lambda weights: weights.sum() - 1,
}

bounds = [(0, 1)] * 4

starts = [
    np.array([0.30, 0.20, 0.38, 0.12]),
    np.array([0.30, 0.10, 0.47, 0.13]),
    np.array([0.25, 0.25, 0.35, 0.15]),
    np.array([0.25, 0.00, 0.60, 0.15]),
    np.array([0.25, 0.15, 0.45, 0.15]),
]

rng = np.random.default_rng(42)

random_starts = rng.dirichlet(
    alpha=np.ones(4),
    size=25,
)

starts.extend(random_starts)

optimization_rows = []

for start_weights in starts:
    result = minimize(
        fun=blend_oof_mape,
        x0=start_weights,
        method="SLSQP",
        bounds=bounds,
        constraints=constraint,
        options={
            "maxiter": 1000,
            "ftol": 1e-10,
        },
    )

    if result.success:
        optimization_rows.append(
            {
                "ridge_weight": result.x[0],
                "v4_weight": result.x[1],
                "v6_weight": result.x[2],
                "text_weight": result.x[3],
                "oof_mape_pct": result.fun,
            }
        )

optimization_results = (
    pd.DataFrame(optimization_rows)
    .drop_duplicates()
    .sort_values("oof_mape_pct")
    .reset_index(drop=True)
)

display(optimization_results.head(20))

,ridge_weight,v4_weight,v6_weight,text_weight,oof_mape_pct
0,0.264182,0.087899,0.549272,0.098647,12.128774
1,0.264182,0.087899,0.549272,0.098647,12.128774
2,0.264182,0.087899,0.549272,0.098648,12.128774
3,0.264182,0.087899,0.549271,0.098648,12.128774
4,0.264182,0.087899,0.549271,0.098648,12.128774
5,0.264182,0.087898,0.549272,0.098647,12.128774
6,0.264182,0.087898,0.549272,0.098647,12.128774
7,0.264182,0.087899,0.549272,0.098648,12.128774
8,0.264182,0.087898,0.549272,0.098647,12.128774
9,0.264182,0.087899,0.549272,0.098647,12.128774


Привести лучший вариант к понятным весам

In [11]:
def round_to_simplex(
    weights: np.ndarray,
    decimals: int = 2,
) -> np.ndarray:
    scale = 10**decimals

    scaled = np.asarray(weights) * scale
    rounded_down = np.floor(scaled).astype(int)

    remaining = int(scale - rounded_down.sum())

    fractional_part = scaled - rounded_down

    add_to = np.argsort(
        -fractional_part
    )[:remaining]

    rounded_down[add_to] += 1

    return rounded_down / scale


best_weights_raw = optimization_results.loc[
    0,
    [
        "ridge_weight",
        "v4_weight",
        "v6_weight",
        "text_weight",
    ],
].to_numpy(dtype=float)

best_weights_simple = round_to_simplex(
    best_weights_raw,
    decimals=2,
)

best_weights_report = pd.DataFrame(
    {
        "model": [
            "Ridge",
            "CatBoost v4",
            "CatBoost v6",
            "Text Ridge",
        ],
        "raw_optimized_weight": best_weights_raw,
        "simple_weight": best_weights_simple,
    }
)

display(best_weights_report)

print(
    "OOF MAPE with raw optimum:",
    f"{blend_oof_mape(best_weights_raw):.3f}%",
)

print(
    "OOF MAPE with simple weights:",
    f"{blend_oof_mape(best_weights_simple):.3f}%",
)

,model,raw_optimized_weight,simple_weight
0,Ridge,0.264182,0.26
1,CatBoost v4,0.087899,0.09
2,CatBoost v6,0.549272,0.55
3,Text Ridge,0.098647,0.10


OOF MAPE with raw optimum: 12.129%
OOF MAPE with simple weights: 12.129%


Проверить устойчивость рядом с оптимумом

In [12]:
def local_weight_search(
    center_weights: np.ndarray,
    radius: int = 3,
) -> pd.DataFrame:
    center = (
        round_to_simplex(
            center_weights,
            decimals=2,
        )
        * 100
    ).astype(int)

    rows = []

    for ridge_count in range(
        max(0, center[0] - radius),
        min(100, center[0] + radius) + 1,
    ):
        for v4_count in range(
            max(0, center[1] - radius),
            min(100, center[1] + radius) + 1,
        ):
            for v6_count in range(
                max(0, center[2] - radius),
                min(100, center[2] + radius) + 1,
            ):
                text_count = (
                    100
                    - ridge_count
                    - v4_count
                    - v6_count
                )

                if not 0 <= text_count <= 100:
                    continue

                if abs(text_count - center[3]) > radius:
                    continue

                weights = np.array(
                    [
                        ridge_count,
                        v4_count,
                        v6_count,
                        text_count,
                    ]
                ) / 100

                rows.append(
                    {
                        "ridge_weight": weights[0],
                        "v4_weight": weights[1],
                        "v6_weight": weights[2],
                        "text_weight": weights[3],
                        "oof_mape_pct": blend_oof_mape(weights),
                    }
                )

    return (
        pd.DataFrame(rows)
        .sort_values("oof_mape_pct")
        .reset_index(drop=True)
    )


local_candidates = local_weight_search(
    best_weights_raw,
    radius=3,
)

display(local_candidates.head(25))

,ridge_weight,v4_weight,v6_weight,text_weight,oof_mape_pct
0,0.26,0.09,0.55,0.10,12.128854
1,0.26,0.08,0.56,0.10,12.128876
2,0.27,0.08,0.55,0.10,12.128941
3,0.27,0.09,0.55,0.09,12.129002
4,0.26,0.10,0.54,0.10,12.129011
5,0.27,0.09,0.54,0.10,12.129043
6,0.26,0.07,0.57,0.10,12.129048
7,0.27,0.08,0.56,0.09,12.129057
8,0.27,0.07,0.56,0.10,12.129067
9,0.27,0.10,0.54,0.09,12.129128


Результат очень хороший и, главное, устойчивый.

Лучший OOF ensemble: 12.129%
CatBoost v6 отдельно: 12.747%
Прирост ансамбля:     −0.618 п.п.

Оптимальные веса получились логичными:

Ridge:       26%
CatBoost v4:  9%
CatBoost v6: 55%
Text Ridge:  10%

И рядом с минимумом много почти одинаковых комбинаций. Это не похоже на случайный «магический» набор коэффициентов.

Что это говорит
v6 — основной двигатель ансамбля.
Обычный Ridge всё ещё очень полезен: 26%.
Text Ridge подтверждён OOF, но его безопасный вес ближе к 10%, а не 20%.
v4 слабее v6 отдельно, но добавляет разнообразие ошибок, поэтому получает свои ~9%.

Но сабмитить этот набор пока рано. Причина: на leaderboard уже подтверждена v5, а в текущем OOF-ансамбле её нет.

Сейчас мы сравнили:

Ridge + v4 + v6 + Text

Но ещё не сравнили главный кандидат:

Ridge + v4 + v5 + v6 + Text

Именно v5 дала лучший реальный leaderboard-результат 13.66%, поэтому выбрасывать её без честной OOF-проверки нельзя.

Следующий правильный шаг — OOF для v5

Это последний дорогой, но действительно оправданный CV-запуск. После него у нас будут честные OOF всех пяти моделей:

Ridge
CatBoost v4
CatBoost v5
CatBoost v6
Text Ridge

И можно будет выбрать финальный ансамбль уже по OOF, а не по одному holdout.

In [ ]:
v5_oof_report = pd.DataFrame(
    {
        "car_id": train["car_id"].to_numpy(),
        "y_true": y.to_numpy(),
        "title_v5_pred": oof_v5_predictions,
        "ape_pct": (
            np.abs(y.to_numpy() - oof_v5_predictions)
            / y.to_numpy()
            * 100
        ),
    }
)

v5_oof_report.to_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v5_oof_predictions.parquet",
    index=False,
)